# 2. Reusable functions, files, and checks
## Goal
Turn a calculation into tested reusable code and read/write CSV and JSON.
Course Modules 3–4. All example data are invented.
## Setup
Standard library only. Files are written to a temporary directory and removed automatically.
Reference: [Python standard library](https://docs.python.org/3/library/).
## Steps
### 1. Define a contract
A function returns a result. Invalid data are rejected, and an empty basket totals zero.

In [1]:
from math import isclose, isfinite
def discounted_total(prices, rate=0.0):
    """Total finite nonnegative prices using a finite discount from 0 to 1."""
    if not isfinite(rate) or not 0 <= rate <= 1:
        raise ValueError("rate must be finite and between 0 and 1")
    if any(not isfinite(price) or price < 0 for price in prices):
        raise ValueError("prices must be finite and nonnegative")
    return sum(prices) * (1 - rate)

print(discounted_total([10, 20], 0.1))

27.0


### 2. Test behavior
Tests include boundaries and invalid values, not only the usual input.

In [2]:
assert isclose(discounted_total([10, 20], 0.1), 27)
assert discounted_total([], 0.5) == 0
assert discounted_total([20], 1) == 0
for prices, rate in [([-1], 0), ([10], 1.5), ([float("nan")], 0)]:
    try:
        discounted_total(prices, rate)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid input was accepted")
print("Normal, boundary, and invalid-input checks passed.")

Normal, boundary, and invalid-input checks passed.


### 3. Read records and write a summary
CSV numeric fields are text until converted. The context managers close files and clean up the temporary folder.

In [3]:
import csv
import json
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as folder:
    source = Path(folder) / "expenses.csv"
    source.write_text("category,amount\nFood,12\nTravel,8\nFood,15\n", encoding="utf-8")
    with source.open(newline="", encoding="utf-8") as handle:
        records = list(csv.DictReader(handle))
    totals = {}
    for record in records:
        category = record["category"]
        totals[category] = totals.get(category, 0) + float(record["amount"])
    output = Path(folder) / "summary.json"
    output.write_text(json.dumps(totals, indent=2), encoding="utf-8")
    restored = json.loads(output.read_text(encoding="utf-8"))
    print(restored)
    assert restored == {"Food": 27.0, "Travel": 8.0}

{'Food': 27.0, 'Travel': 8.0}


### 4. Parse an API-shaped response
This is a local string, not a live network request. Validate the structure you rely on.

In [4]:
payload = '{"status": "ok", "items": [{"id": 1, "amount": 12}]}'
response = json.loads(payload)
if response.get("status") != "ok" or not isinstance(response.get("items"), list):
    raise ValueError("Unexpected response shape")
print(response["items"][0]["amount"])

12


## Checks
All assertions should pass from a fresh kernel. Explain why the `except` block is narrow and why JSON parsing alone does not validate business meaning.
## Next Steps
Complete exercises 11–15. Your CSV mini-project should preserve raw data, record rejected rows, and handle a new file. Add meaningful tests rather than copying the function's implementation into a test.